# Make an RHEF movie of PUNCH data

Download PUNCH coronagraph mosaics, sharpen them with the **Radial Histogram
Equalizing Filter (RHEF)**, and save a movie — in a handful of cells.

RHEF flattens the steep radial brightness drop-off of the corona so faint outer
structure (streamers, CME fronts) shows up as clearly as the bright inner corona.

Run the cells top to bottom — the first one installs everything it needs.
**Requires Python ≥ 3.12** (that's the floor for `sunkit-image >= 0.7`, which
adds RHEF's `upsilon` knob). If your kernel is older, start Jupyter from a
Python 3.12+ environment or pick a 3.12+ kernel.

_Paper:_ [Gilly & Cranmer 2025](https://link.springer.com/article/10.1007/s11207-025-02578-x) ·
_API:_ [`sunkit_image.radial.rhef`](https://docs.sunpy.org/projects/sunkit-image/en/stable/api/sunkit_image.radial.rhef.html)

## Setup

Install the packages into the current kernel. Safe to skip if you already have them.

In [ ]:
import sys
assert sys.version_info >= (3, 12), (
    "This example needs Python >= 3.12 (sunkit-image >= 0.7, which adds RHEF's "
    f"upsilon knob). This kernel is Python {sys.version.split()[0]} — start Jupyter "
    "from a Python 3.12+ environment, or pick a 3.12+ kernel."
)

%pip install -q sunpy "sunkit-image>=0.7" punchbowl imageio-ffmpeg ipywidgets

import sunkit_image
from packaging.version import Version
assert Version(sunkit_image.__version__) >= Version("0.7"), "need sunkit-image >= 0.7 for the upsilon knob"
print("ready · Python", sys.version.split()[0], "· sunkit-image", sunkit_image.__version__)

## 1. Get the data

Search and download PUNCH level-3 mosaics with sunpy's `Fido`. PUNCH's Fido
client splits the product name into a 2-letter `ProductCode` (`CT` = clear
trefoil) plus an `Instrument` (`M` = mosaic) — so `CTM` = `CT` + `M`. Each
mosaic is ~17 MB, so we grab `n` of them spread across one day.

In [ ]:
import logging, warnings
logging.getLogger('parfive').setLevel(logging.ERROR)   # quiet the downloader
warnings.filterwarnings("ignore", message="This download has been started in a thread")
import punchbowl                                    # registers the PUNCH Fido client
from concurrent.futures import ThreadPoolExecutor
from sunpy.net import Fido, attrs as a

def get_punch(day="2026-05-11", n=4, out="punch_data"):
    """Search + download n PUNCH L3 CTM mosaics for a UTC day via Fido."""
    results = Fido.search(a.Time(f"{day} 00:00", f"{day} 23:59"),
                          a.Source("PUNCH"), a.Level("3"),
                          a.punch.ProductCode("CT"), a.Instrument("M"))
    rows = results[0][:: max(1, len(results[0]) // n)][:n]   # spread n frames across the day

    def fetch():                                   # run off Jupyter's event loop:
        files = Fido.fetch(rows, path=f"{out}/{{file}}", progress=False)   # avoids a noisy
        for _ in range(4):                         # parfive/asyncio teardown warning, and
            if not files.errors:                   # re-fetches any files the server dropped
                break
            files = Fido.fetch(files, progress=False)
        return files
    with ThreadPoolExecutor(1) as pool:
        files = pool.submit(fetch).result()
    if not files:
        raise RuntimeError("No PUNCH files downloaded — likely a transient network "
                           "hiccup. Just run this cell again.")
    return sorted(map(str, files))

files = get_punch(n=4)
print(f"{len(files)} frames ready")

## 2. Look at the raw data

Load one mosaic and look at it unfiltered. Coronal brightness drops so steeply
with radius that a plain linear scale shows essentially nothing; clip it and the
inner corona blows out white while the outer field sinks to black; a log stretch
gets you structure but leaves the outer corona flat. That dynamic range is the
problem RHEF solves.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import astropy.units as u
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import sunpy.map

SIZE = 1024   # working resolution — RHEF's cost scales with pixel count, and 1024²
              # is plenty for figures and movies. Use size=None for the native 4096².

def load(path, size=SIZE):
    "Open a PUNCH mosaic as a sunpy Map (the FITS has several HDUs)."
    m = sunpy.map.Map(path)[0]
    return m.resample([size, size] * u.pix) if size else m

m = load(files[0])

lo, hi = np.nanpercentile(m.data, [1, 99.5])
fig, axes = plt.subplots(1, 2, figsize=(13, 6.5))
axes[0].imshow(m.data, origin="lower", cmap="punch", vmin=lo, vmax=hi)
axes[0].set_title("raw — 1–99.5% clip")
axes[1].imshow(m.data, origin="lower", cmap="punch", norm=LogNorm())
axes[1].set_title("raw — log scale")
for ax in axes: ax.axis("off")
plt.tight_layout(); plt.show()

## 3. Filter one frame

Now the whole idea: **call `rhef`.** No tuning needed — and the PUNCH colormap
comes along for free. Compare this to the raw frames above.

In [ ]:
from sunkit_image.radial import rhef

filtered = rhef(m)     # ← the filter

fig = plt.figure(figsize=(13, 6.5))

ax1 = fig.add_subplot(121, projection=m)
m.plot(axes=ax1, clip_interval=(5, 99.95) * u.percent)   # raw needs a stretch to show anything
ax1.set_title("raw")

ax2 = fig.add_subplot(122, projection=filtered)
filtered.plot(axes=ax2)                                  # mission colormap, built in
ax2.set_title("RHEF")

plt.show()

## 3.5 Scrub between before and after

Drag the slider to sweep the divider across the frame — RHEF on the left of the
line, raw on the right. The raw side is percentile-clipped onto the same 0–1
scale so the two halves are directly comparable.

Every slider position is rendered once up front (about a tenth of a second), so
dragging is instant. Note that this needs a live kernel; in a statically
rendered page it shows a single frame rather than a working slider.

In [ ]:
import ipywidgets as widgets
from IPython.display import display
from io import BytesIO
from PIL import Image as PILImage

VIEW = 700                                  # display size, px

cmap = plt.get_cmap("punch")
lo, hi = np.nanpercentile(m.data, [1, 99.5] * u.percent)

def to_rgb(arr):
    """Colour-map once, up front, so scrubbing never touches matplotlib."""
    rgb = (cmap(arr)[..., :3] * 255).astype(np.uint8)[::-1]      # [::-1] = origin="lower"
    return np.asarray(PILImage.fromarray(rgb).resize((VIEW, VIEW)))

rgb_rhef = to_rgb(filtered.data)
rgb_raw  = to_rgb(np.clip((m.data - lo) / (hi - lo), 0, 1))

def compose(x):
    combo = np.concatenate([rgb_raw[:, :x], rgb_rhef[:, x:]], axis=1)
    combo[:, max(0, x - 2):x + 1] = 255                          # the divider line
    buf = BytesIO()
    PILImage.fromarray(combo).save(buf, format="JPEG", quality=85)
    return buf.getvalue()

frames = [compose(round(VIEW * s / 100)) for s in range(101)]     # ~0.1 s, then scrubbing is free

image  = widgets.Image(value=frames[50], format="jpeg",
                       layout=widgets.Layout(width=f"{VIEW}px", height=f"{VIEW}px"))
slider = widgets.IntSlider(value=50, min=0, max=100, readout=False, continuous_update=True,
                           layout=widgets.Layout(width=f"{VIEW}px", margin="0px"))
slider.observe(lambda ch: setattr(image, "value", frames[ch["new"]]), names="value")

labels = widgets.HTML(f'<div style="display:flex;justify-content:space-between;'
                      f'width:{VIEW}px;font-family:monospace">'
                      f'<span>&#9664; before RHEF</span><span>after RHEF &#9654;</span></div>')
display(widgets.VBox([image, slider, labels],
                     layout=widgets.Layout(width=f"{VIEW}px")))

## 4. The one knob: Upsilon (Υ)

`upsilon` sets how hard RHEF pushes the contrast: `None` is maximum (flat)
equalization, smaller is gentler, and the default `0.35` sits in between.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 5))
for ax, up in zip(axes, [None, 1.0, 0.35, 0.1]):
    img = rhef(m, upsilon=(up))
    ax.imshow(img.data, origin="lower", cmap="punch", vmin=0, vmax=1)
    ax.set_title(f"upsilon = {up}"); ax.axis("off")
plt.show()

## 5. `radial_bin_edges`: leave it out (almost always)

RHEF equalizes the pixels *within each radial annulus*, so the annuli want to be
about **one pixel wide**. Omit the keyword and `rhef` sizes them for you — one
annulus per pixel radius (a 4096² image gets 2048 bins; ≈1.5 px here). Set them
any coarser and each ring equalizes to its own slightly different level, which
shows up as **concentric banding** — the classic RHEF artifact, made deliberately
in the cell below so you can recognise it.

The one real tradeoff: narrow annuli are more susceptible to outliers. Off-limb
that rarely matters — but if you see *dark* banding at the same radius as a very
bright feature, widening the annuli is the fix. That's the good reason to pass
`radial_bin_edges` yourself.

In [ ]:
from sunkit_image.utils import equally_spaced_bins

coarse = equally_spaced_bins(0, 174, 40) * u.R_sun     # 40 fat annuli ≈ 13 px wide

fig, axes = plt.subplots(1, 2, figsize=(13, 6.5))
for ax, (img, title) in zip(axes, [(rhef(m), "rhef(m) — default bins ≈1.5 px"),
                                   (rhef(m, radial_bin_edges=coarse), "coarse bins — banding")]):
    ax.imshow(img.data, origin="lower", cmap="punch", vmin=0, vmax=1)
    ax.set_title(title); ax.axis("off")
plt.tight_layout(); plt.show()

## 6. Make the movie

Filter every frame, save a PNG of each, and stitch them into an MP4 with the
`ffmpeg` bundled in `imageio-ffmpeg` (nothing to install system-wide).

In [ ]:
import subprocess, imageio_ffmpeg
from pathlib import Path

FRAMES = Path("frames"); FRAMES.mkdir(exist_ok=True)
for old in FRAMES.glob("*.png"): old.unlink()

pngs = []
for f in files:
    try:
        frame = load(f)
        img = rhef(frame)
    except Exception as e:
        print("skip", Path(f).name, e); continue
    fig = plt.figure(figsize=(6, 6)); ax = fig.add_axes([0, 0, 1, 1])
    ax.imshow(img.data, origin="lower", cmap="punch", vmin=0, vmax=1); ax.axis("off")
    ax.text(0.02, 0.02, frame.date.iso[:19], color="white", family="monospace",
            fontsize=9, transform=ax.transAxes)
    out = FRAMES / f"{len(pngs):03d}.png"
    fig.savefig(out, dpi=130); plt.close(fig); pngs.append(out)

ffmpeg = imageio_ffmpeg.get_ffmpeg_exe()
subprocess.run([ffmpeg, "-y", "-framerate", "8", "-i", str(FRAMES / "%03d.png"),
                "-c:v", "libx264", "-pix_fmt", "yuv420p",
                "-vf", "scale=trunc(iw/2)*2:trunc(ih/2)*2", "punch_rhef.mp4"],
               check=True, capture_output=True)
print(f"wrote punch_rhef.mp4 ({len(pngs)} frames)")

## 7. Play it

In [ ]:
from IPython.display import Video
Video("punch_rhef.mp4", embed=True, width=480)

## 8. One nice figure

A publication-style frame: solar-distance rings, the mission colormap, and a
calibrated colorbar.

In [ ]:
import numpy as np
from matplotlib.patches import Circle

r_max = (m.data.shape[1] / 2) * m.scale[0].to_value(u.arcsec / u.pix) / m.rsun_obs.to_value(u.arcsec)
data = rhef(m).data

with plt.rc_context({"font.family": "monospace"}):
    fig, ax = plt.subplots(figsize=(7.5, 7.5), facecolor="#0c0a07")
    im = ax.imshow(data, origin="lower", extent=[-r_max, r_max, -r_max, r_max],
                   cmap="punch", vmin=0, vmax=1)
    ax.set_facecolor("black")
    for rr in (50, 100, 150):                          # solar-distance rings
        ax.add_patch(Circle((0, 0), rr, fill=False, ec="#f3ead9", lw=0.5, alpha=0.25))
        ax.text(0, -rr, str(rr), color="#f3ead9", fontsize=8, alpha=0.6, ha="center", va="center",
                bbox=dict(boxstyle="round,pad=0.1", fc="#0c0a07", ec="none", alpha=0.6))
    ax.set_xlim(-r_max, r_max); ax.set_ylim(-r_max, r_max)
    for s in ax.spines.values(): s.set_color("#2a2318")
    ax.tick_params(colors="#a8967a", labelsize=8)
    ax.set_xlabel(r"solar-X  [R$_\odot$]", color="#a8967a")
    ax.set_ylabel(r"solar-Y  [R$_\odot$]", color="#a8967a")
    ax.set_title(rf"PUNCH L3 CTM   ·   RHEF $\Upsilon$=0.35   ·   {m.date.iso[:16]} UTC",
                 color="#f3ead9", fontsize=11, pad=12)
    cb = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
    cb.set_label("RHEF equalized brightness", color="#a8967a", fontsize=9)
    cb.ax.yaxis.set_tick_params(color="#a8967a", labelsize=8)
    cb.outline.set_edgecolor("#2a2318")
    plt.setp(plt.getp(cb.ax, "yticklabels"), color="#a8967a")
plt.show()